# DiffusionGemma vs Gemma 4 Colab Runner

This notebook is optimized for VS Code with a Google Colab kernel. The first experiment clones the benchmark repository from GitHub, runs local harness tests, executes preflight and a placeholder smoke phase, packages small result artifacts, and optionally commits them to a `bench-results` branch.

## 0. Experiment Settings

Set `REPO_URL` before running the clone cell. Use a normal HTTPS GitHub URL for public repos. For private repos, authenticate through VS Code/Colab secrets or use a token-backed URL only inside the runtime, never in committed notebook output.

In [ ]:
REPO_URL = ""  # example: "https://github.com/<owner>/diffusion_gemma_bench.git"
CODE_BRANCH = "main"
RESULTS_BRANCH = "bench-results"
PROFILE = "q6_core_native"
PHASE = "smoke"
PROJECT_DIR = "/content/diffusion_gemma_bench"

assert REPO_URL, "Set REPO_URL before continuing."

## 1. Runtime Check

This confirms that the VS Code notebook is executing on the selected Colab runtime and shows the assigned GPU.

In [ ]:
import pathlib, platform, subprocess, sys

def run(cmd, cwd=None, check=False):
    print(f"$ {' '.join(cmd)}")
    completed = subprocess.run(cmd, cwd=cwd, text=True, capture_output=True)
    if completed.stdout:
        print(completed.stdout)
    if completed.stderr:
        print(completed.stderr)
    if check and completed.returncode != 0:
        raise RuntimeError(f"command failed: {' '.join(cmd)}")
    return completed

print('python:', sys.version)
print('platform:', platform.platform())
print('cwd:', pathlib.Path.cwd())
print('content_exists:', pathlib.Path('/content').exists())
run(['nvidia-smi'])

## 2. Clone Code From GitHub

This keeps the Colab experiment tied to a real Git commit. The commit SHA is written into `results/preflight.json` and `results/run_manifest.json` by the harness.

In [ ]:
import os, pathlib, shutil, sys

project_path = pathlib.Path(PROJECT_DIR)
if project_path.exists():
    shutil.rmtree(project_path)

run(['git', 'clone', '--branch', CODE_BRANCH, '--depth', '1', REPO_URL, PROJECT_DIR], check=True)
os.chdir(PROJECT_DIR)
if PROJECT_DIR not in sys.path:
    sys.path.insert(0, PROJECT_DIR)

CODE_COMMIT_SHA = run(['git', 'rev-parse', 'HEAD'], cwd=PROJECT_DIR, check=True).stdout.strip()
CODE_BRANCH_ACTUAL = run(['git', 'rev-parse', '--abbrev-ref', 'HEAD'], cwd=PROJECT_DIR, check=True).stdout.strip()
print('PROJECT_DIR:', PROJECT_DIR)
print('CODE_BRANCH:', CODE_BRANCH_ACTUAL)
print('CODE_COMMIT_SHA:', CODE_COMMIT_SHA)
run(['git', 'status', '--short'], cwd=PROJECT_DIR, check=True)

## 3. Harness Smoke Test

This does not download model weights yet. It verifies repository integrity, basic result writing, GPU/resource preflight, report generation, and the placeholder smoke gate.

In [ ]:
checks = [
    [sys.executable, '-m', 'unittest', 'discover', '-s', 'tests'],
    [sys.executable, 'run.py', '--profile', 'auto', '--phase', 'preflight'],
    [sys.executable, 'run.py', '--profile', PROFILE, '--phase', PHASE],
    [sys.executable, 'run.py', '--profile', PROFILE, '--phase', 'report'],
]

for cmd in checks:
    completed = run(cmd, cwd=PROJECT_DIR)
    if completed.returncode != 0:
        raise SystemExit(completed.returncode)

## 4. Inspect Result Artifacts

`smoke_status.json` should currently say `PENDING_COLAB_BACKEND_GATE`. That is expected until the real vLLM model smoke gate is implemented.

In [ ]:
import json, pathlib

for path in ['results/preflight.json', 'results/run_manifest.json', 'results/smoke_status.json', 'reports/final_report.md']:
    p = pathlib.Path(PROJECT_DIR) / path
    print('\n###', path)
    if p.suffix == '.json':
        print(json.dumps(json.loads(p.read_text()), indent=2, ensure_ascii=False)[:5000])
    else:
        print(p.read_text(encoding='utf-8')[:2000])

## 5. Package Result Run

This copies small, reviewable files into `results/runs/<run_id>/` and validates that no model weights, large logs, or secret-looking tokens are present.

In [ ]:
from src.result_store import make_run_id, package_results, validate_result_tree

RUN_ID = make_run_id(profile=PROFILE, phase=PHASE, commit_sha=CODE_COMMIT_SHA)
manifest = package_results(run_id=RUN_ID, profile=PROFILE, phase=PHASE)
run_dir = pathlib.Path(PROJECT_DIR) / 'results' / 'runs' / RUN_ID
validation = validate_result_tree(run_dir)
print('RUN_ID:', RUN_ID)
print('run_dir:', run_dir)
print('copied_files:', len(manifest['copied_files']))
print('validation:', validation)
assert validation['ok'], validation

## 6. Optional: Commit Results To GitHub

Run this only after inspecting the packaged run. For private repos, authenticate GitHub in the runtime first. Recommended: use Colab secrets or a temporary token, and never print the token. The script commits only `results/runs/<RUN_ID>` after safety validation.

In [ ]:
PUSH_RESULTS = False
GIT_USER_NAME = "Colab Benchmark Bot"
GIT_USER_EMAIL = "colab-benchmark@example.invalid"

if PUSH_RESULTS:
    run(['git', 'config', 'user.name', GIT_USER_NAME], cwd=PROJECT_DIR, check=True)
    run(['git', 'config', 'user.email', GIT_USER_EMAIL], cwd=PROJECT_DIR, check=True)
    run([
        sys.executable,
        'scripts/push_results_to_github.py',
        '--profile', PROFILE,
        '--phase', PHASE,
        '--run-id', RUN_ID,
        '--branch', RESULTS_BRANCH,
        '--commit',
        '--push',
    ], cwd=PROJECT_DIR, check=True)
else:
    print('PUSH_RESULTS is False. Packaged results are ready for manual inspection.')

## 7. Next Step After This Smoke

If this notebook works end to end, the next repository change is the real vLLM capability gate: install/check vLLM, verify HF access, start one local server, test health/streaming/tool JSON, and only then attempt model-specific smoke tests.